# Which target does this drug hit?Cancer cells were treated with a drug. The L1000 assay then measured how 978 genesresponded. That response is the drug's **fingerprint**.Drugs that hit the same target tend to leave similar fingerprints. Your job: read thefingerprints of ~30 unknown drugs and say which target class each one hits.**This notebook takes you from zero to a scored submission.** Run the cells in order.Nothing here needs a GPU and nothing takes more than a few seconds.| Step | What you do || --- | --- || 1 | Load the data and look at it || 2 | Look at one drug's fingerprint || 3 | Build an "average fingerprint" per target class || 4 | Classify by similarity (your first model) || 5 | Check how good it is, honestly || 6 | Predict the unknown drugs and save a submission || 7 | Ideas for what to try next |

## Step 1 - Load the dataFour files matter. Everything is a plain table; `pandas` reads them directly.

In [1]:
import pandas as pd
import numpy as np
DATA = ".."        # change if you copied the data somewhere else
X_train = pd.read_csv(f"{DATA}/X_train.csv.gz", index_col=0)   
# fingerprint 
y_train = pd.read_csv(f"{DATA}/y_train.csv", index_col=0)["moa"]  
# the answer
meta = pd.read_csv(f"{DATA}/meta_train.csv", index_col=0)   
# compound, dose, cv_fold
X_test  = pd.read_csv(f"{DATA}/X_test.csv.gz", index_col=0)    
# the unknown drugs
print("training fingerprints:", X_train.shape, "= rows x genes")
print("unknown drugs:", X_test.shape)
print()
print("target classes:")
print(y_train.value_counts())

training fingerprints: (815, 978) = rows x genes
unknown drugs: (174, 978)

target classes:
moa
proteasome inhibitor                318
topoisomerase inhibitor              66
EGFR inhibitor                       60
HDAC inhibitor                       54
tubulin polymerization inhibitor     47
MEK inhibitor                        42
CDK inhibitor                        36
JAK inhibitor                        24
HSP inhibitor                        24
PARP inhibitor                       24
mTOR inhibitor                       24
RAF inhibitor                        24
AKT inhibitor                        24
PI3K inhibitor                       24
PLK inhibitor                        24
Name: count, dtype: int64


In [2]:
pwd

'/Users/amirreza/PythonProjects/bioHackaton2026/starter_kit'

**Read the shapes above.** Each row is one measurement: one drug at one dose. Each columnis one gene. A value is how much that gene changed compared to untreated cells: positivemeans it went up, negative means it went down, and roughly anything beyond +2 or -2 is astrong change.Note that the same drug appears several times, once per dose:

In [3]:
meta.head(8)

,compound,dose_um,cv_fold
sample_id,,,
REP.A001_A549_24H:D07,BRD-K26011976,10.00,1
REP.A001_A549_24H:D08,BRD-K26011976,3.33,1
REP.A001_A549_24H:D09,BRD-K26011976,1.11,1
REP.A001_A549_24H:D10,BRD-K26011976,0.37,1
REP.A001_A549_24H:D11,BRD-K26011976,0.12,1
REP.A001_A549_24H:D12,BRD-K26011976,0.04,1
REP.A001_A549_24H:E01,BRD-K26603252,10.00,1
REP.A001_A549_24H:E03,BRD-K26603252,1.11,1


## Step 2 - Look at a single fingerprintBefore modelling anything, look at the data. Pick one drug and see which genes moved most.

In [4]:
# pick any training row
row_id = X_train.index[0]
compound = meta.loc[row_id, "compound"]
print(f"row {row_id}  | compound {compound}  |  true class: {y_train[row_id]}")
fingerprint = X_train.loc[row_id]
print("\nmost increased genes:")
print(fingerprint.nlargest(10).round(2))
print("\nmost decreased genes:")
print(fingerprint.nsmallest(10).round(2))

row REP.A001_A549_24H:D07  | compound BRD-K26011976  |  true class: topoisomerase inhibitor

most increased genes:
202847_at      7.51
201202_at      6.90
202555_s_at    6.14
201397_at      4.70
208763_s_at    4.46
201761_at      4.21
203665_at      4.20
219573_at      3.67
204587_at      3.35
202887_s_at    3.31
Name: REP.A001_A549_24H:D07, dtype: float64

most decreased genes:
200799_at     -5.19
201849_at     -3.88
203277_at     -3.79
212115_at     -3.69
209095_at     -3.60
221750_at     -3.48
208398_s_at   -3.47
212694_s_at   -3.46
203007_x_at   -3.44
212955_s_at   -3.42
Name: REP.A001_A549_24H:D07, dtype: float64


**Try this:** change `X_train.index[0]` to another row and compare. Two rows of the samedrug at different doses should look related; two unrelated drugs should not.If you have a biology background, this is where you earn your keep: do these genes makesense for that target class? Write down what you expect *before* you model anything -it makes a much better story in your presentation.

## Step 3 - An average fingerprint per classThe simplest useful idea: if drugs hitting the same target look alike, then the *average*fingerprint of a class is a decent description of that class.

In [5]:
class_profiles = X_train.groupby(y_train).mean()
print(class_profiles.shape, "= classes x genes")# the genes that most define one class
cls = class_profiles.index[0]
print(f"\nclass: {cls}")
print("up:  ", list(class_profiles.loc[cls].nlargest(8).index))
print("down:", list(class_profiles.loc[cls].nsmallest(8).index))

(15, 978) = classes x genes

class: AKT inhibitor
up:   ['212665_at', '208891_at', '205039_s_at', '217766_s_at', '202458_at', '209576_at', '201125_s_at', '208936_x_at']
down: ['201397_at', '203119_at', '208708_x_at', '201761_at', '202887_s_at', '201351_s_at', '209095_at', '211071_s_at']


## Step 4 - Your first model: classify by similarityCompare a new fingerprint to each class profile and pick the closest one. The similaritymeasure is cosine similarity: it asks whether two fingerprints point in the samedirection, ignoring how strong they are.This is not a toy. It is essentially how the Connectivity Map works, and it is one of thetwo official baselines.

In [6]:
def unit(M):    
    """Scale every row to length 1 so we can compare directions."""    
    M = np.asarray(M, dtype=float)    
    return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)

def classify(fingerprints, profiles):    
    """Return a similarity score of every fingerprint to every class."""    
    sim = unit(fingerprints[profiles.columns]) @ unit(profiles).T    
    return pd.DataFrame(sim, index=fingerprints.index, columns=profiles.index)
    
scores = classify(X_train, class_profiles)
predicted = scores.idxmax(axis=1)
print("first few predictions vs truth:")
pd.DataFrame({"predicted": predicted, "true": y_train}).head(20)

first few predictions vs truth:


,predicted,true
sample_id,,
REP.A001_A549_24H:D07,RAF inhibitor,topoisomerase inhibitor
REP.A001_A549_24H:D08,JAK inhibitor,topoisomerase inhibitor
REP.A001_A549_24H:D09,JAK inhibitor,topoisomerase inhibitor
REP.A001_A549_24H:D10,PARP inhibitor,topoisomerase inhibitor
REP.A001_A549_24H:D11,EGFR inhibitor,topoisomerase inhibitor
REP.A001_A549_24H:D12,PI3K inhibitor,topoisomerase inhibitor
REP.A001_A549_24H:E01,EGFR inhibitor,EGFR inhibitor
REP.A001_A549_24H:E03,EGFR inhibitor,EGFR inhibitor
REP.A001_A549_24H:E05,EGFR inhibitor,EGFR inhibitor


## Step 5 - How good is it, really?The numbers above are dishonest: we scored the model on the same data we built it from.Two rules make the evaluation honest, and they are the rules the judges care about:1. **Never test on data you trained on.** Use the `cv_fold` column: train on folds 1-3,   test on fold 0, and so on.2. **Keep a drug together.** All doses of one drug are in the same fold already, because   otherwise the model just recognises the drug instead of the mechanism.

In [7]:
from sklearn.metrics import f1_score

honest_predictions = pd.Series(index=X_train.index, dtype=object)
for fold in sorted(meta["cv_fold"].unique()):    
    held_out = meta["cv_fold"] == fold    
    profiles = X_train[~held_out].groupby(y_train[~held_out]).mean()    
    honest_predictions[held_out] = classify(X_train[held_out], profiles).idxmax(axis=1)
    

    # score per DRUG, not per row: average the doses first
drug_truth = y_train.groupby(meta["compound"]).first()
drug_pred  = honest_predictions.groupby(meta["compound"]).agg(lambda s: s.mode()[0])
macro_f1 = f1_score(drug_truth, drug_pred.loc[drug_truth.index], average="macro")
accuracy = (drug_pred.loc[drug_truth.index] == drug_truth).mean()
print(f"macro-F1 : {macro_f1:.3f}   <- the official metric")
print(f"accuracy : {accuracy:.3f}")

macro-F1 : 0.645   <- the official metric
accuracy : 0.667


**macro-F1** averages the score of each class separately, so a rare class counts as muchas a common one. Guessing the biggest class every time gives a terrible macro-F1, which isthe point.Which classes does it get wrong? This table is often the most interesting result of thewhole day:

In [8]:
confusion = pd.crosstab(drug_truth, drug_pred.loc[drug_truth.index], rownames=["true"], colnames=["predicted"])
confusion

predicted,AKT inhibitor,CDK inhibitor,EGFR inhibitor,HDAC inhibitor,HSP inhibitor,JAK inhibitor,MEK inhibitor,PARP inhibitor,PI3K inhibitor,PLK inhibitor,RAF inhibitor,mTOR inhibitor,proteasome inhibitor,topoisomerase inhibitor,tubulin polymerization inhibitor
true,,,,,,,,,,,,,,,
AKT inhibitor,2,0,0,0,0,0,0,1,1,0,0,0,0,0,0
CDK inhibitor,0,4,0,0,0,0,1,0,0,1,0,0,0,0,0
EGFR inhibitor,1,0,7,0,0,0,0,0,1,0,0,1,0,0,0
HDAC inhibitor,2,1,0,5,0,0,0,0,1,0,0,0,0,0,0
HSP inhibitor,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0
JAK inhibitor,0,0,0,0,0,1,0,0,2,0,1,0,0,0,0
MEK inhibitor,0,0,0,0,0,0,7,0,0,0,0,0,0,0,0
PARP inhibitor,0,0,0,0,0,2,0,1,1,0,0,0,0,0,0
PI3K inhibitor,1,0,0,0,0,0,0,0,3,0,0,0,0,0,0


**Look at the off-diagonal entries and ask why.** Some confusions are modelling noise.Some are real biology: two targets in the same pathway genuinely do similar things to acell, and no amount of machine learning will separate them cleanly. Telling those twoapart, with evidence, is exactly the kind of insight the judges are looking for.

## Step 6 - Predict the unknown drugs and save a submissionSame model, now applied to the drugs whose class we are not told.

In [9]:
meta_test = pd.read_csv(f"{DATA}/meta_test.csv", index_col=0)
test_scores = classify(X_test, class_profiles)            # built from ALL training data
drug_scores = test_scores.groupby(meta_test["compound"]).mean()   # average the doses
drug_scores.index.name = "compound"
drug_scores.reset_index().to_csv("submission.csv", index=False)
print(f"wrote submission.csv: {len(drug_scores)} unknown drugs")
drug_scores.idxmax(axis=1).head(10)


wrote submission.csv: 29 unknown drugs


compound
BRD-A03506276                       HSP inhibitor
BRD-A33280134             topoisomerase inhibitor
BRD-A36331462                proteasome inhibitor
BRD-K02389548                      HDAC inhibitor
BRD-K05804044                      PI3K inhibitor
BRD-K11153516    tubulin polymerization inhibitor
BRD-K12423485                       CDK inhibitor
BRD-K13154216                      mTOR inhibitor
BRD-K13390322                       CDK inhibitor
BRD-K23677682                       JAK inhibitor
dtype: object

That file is a valid submission. You now have a result, and the rest of the hackathon isabout making it better and understanding it.## Step 7 - Where to go nextPick whatever matches your team's interests. None of these need a GPU.**Data and biology**- Low doses often barely change the cells. Count how many genes moved per row  (`(X_train.abs() > 2).sum(axis=1)`) and try dropping the quiet rows before averaging.- Instead of averaging doses equally, weight the strong signatures more.- Score known pathway gene sets instead of individual genes, and see whether the classes  separate better.**Modelling**- Replace the similarity rule with `LogisticRegression` or `RandomForestClassifier` from  scikit-learn. Keep the same honest `cv_fold` evaluation.- `X_unlabeled.csv.gz` holds ~1,000 extra drugs with no labels. Use them to find the main  axes of variation (PCA) and feed those into your model.**Interpretation**- For each class, list the genes that drive the prediction and check them against  pathway knowledge.- Find a class the model keeps failing on and work out whether the problem is the data,  the model, or the biology.**A note on the test set.** It holds only ~30 drugs, so a difference of a few percent isnoise, not progress. If two of your models score within a few points of each other, sayso in your presentation. Judges reward honest reporting over a confident number.